# Binary Search
Binary search works when the **search space is monotonic**.

Monotonic means:

- either **increasing**
- or **decreasing**
- or can be converted into a **boolean monotonic function**

Binary search finds the **boundary where the transition happens**.

## Category 1 — The Duplicate Pivot: From Search X → Bounds → Occurrences

---

## The Core Invariant Shift

**Search X (unique):** The moment `arr[mid] == target`, you're done. The search space collapses to a point.

**With duplicates**, `arr[mid] == target` is *not* terminal — it's a **candidate**. The invariant becomes:

> *"Shrink the search space while preserving the answer inside it."*

This single shift changes `return mid` → `ans = mid; continue searching`.

---

## Lower Bound — $\text{lb}(x)$

Find the **leftmost index** where $arr[i] \geq x$.

**Search Space Invariant:**
$$\forall j < \text{low}: arr[j] < x \qquad \forall j \geq \text{high}: arr[j] \geq x$$

**Boundary Transitions:**

| Condition | Action | Why |
|---|---|---|
| $arr[mid] \geq x$ | `ans = mid; high = mid - 1` | `mid` is a valid candidate, but go left for a *better* (earlier) one |
| $arr[mid] < x$ | `low = mid + 1` | Everything ≤ `mid` is strictly eliminated |

**The delta:** You never do `low = mid` here because `arr[mid] < x` means `mid` itself is dead — safe to do `mid + 1`.

---

## Upper Bound — $\text{ub}(x)$

Find the **leftmost index** where $arr[i] > x$.

Flip one comparison: `arr[mid] > x` → candidate, go left. `arr[mid] <= x` → go right.

$$\text{ub}(x) - \text{lb}(x) = \text{count}(x)$$

---

## First Occurrence

Identical to Lower Bound **but** the candidate condition is `arr[mid] == x` only (not `>=`).

$$\text{first}(x) = \text{lb}(x) \text{ iff } arr[\text{lb}(x)] == x$$

**Boundary Transitions:**

| Condition | Action |
|---|---|
| $arr[mid] == x$ | `ans = mid; high = mid - 1` |
| $arr[mid] > x$ | `high = mid - 1` |
| $arr[mid] < x$ | `low = mid + 1` |

The first two rows can be merged: `arr[mid] >= x → high = mid - 1` (with candidate save on equality). This is exactly `lb`.

---

## Last Occurrence

Mirror of first — candidate on `arr[mid] == x`, but push **right**.

| Condition | Action |
|---|---|
| $arr[mid] == x$ | `ans = mid; low = mid + 1` |
| $arr[mid] < x$ | `low = mid + 1` |
| $arr[mid] > x$ | `high = mid - 1` |

Merge rows 1 & 2: `arr[mid] <= x → low = mid + 1` (with candidate save). This is `ub(x) - 1`.

---

## The Unified Mental Model

```
lb(x)   →  first index where arr[i] >= x   →  candidate when >=, eliminate when <
ub(x)   →  first index where arr[i] >  x   →  candidate when  >, eliminate when <=
first   →  lb(x), validate arr[lb] == x
last    →  ub(x) - 1, validate arr[ub-1] == x
count   →  ub(x) - lb(x)
```

---

## Key Implementation Notes

- `ans` initialized to `arr.size()` (not `-1`) — STL convention, correctly handles *"x larger than all elements"* without a separate check.
- `firstOccurrence` and `lastOccurrence` are **reductions of lb/ub**, not independent implementations.
- `lo + (hi - lo) / 2` over `(lo + hi) / 2` — overflow guard, non-negotiable.

In [1]:
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

In [2]:
// --- Lower Bound: first index where arr[i] >= x ---
int lowerBound(const vector<int>& arr, int x) {
    int n = (int)arr.size();
    int lo = 0, hi = n - 1, ans = n; 
    while (lo <= hi) {
        int mid = lo + (hi - lo) / 2;
        if (arr[mid] >= x) { 
            ans = mid; 
            hi = mid - 1; 
        } else { 
            lo = mid + 1; 
        }
    }
    return ans;
}

// --- Upper Bound: first index where arr[i] > x ---
int upperBound(const vector<int>& arr, int x) {
    int n = (int)arr.size();
    int lo = 0, hi = n - 1, ans = n;
    while (lo <= hi) {
        int mid = lo + (hi - lo) / 2;
        if (arr[mid] > x) { 
            ans = mid; 
            hi = mid - 1; 
        } else { 
            lo = mid + 1; 
        }
    }
    return ans;
}

// --- First Occurrence ---
int firstOccurrence(const vector<int>& arr, int x) {
    int idx = lowerBound(arr, x);
    if (idx == (int)arr.size() || arr[idx] != x) return -1;
    return idx;
}

// --- Last Occurrence ---
int lastOccurrence(const vector<int>& arr, int x) {
    int idx = upperBound(arr, x) - 1;
    // Check if idx is valid and actually contains the target
    if (idx < 0 || idx >= (int)arr.size() || arr[idx] != x) return -1;
    return idx;
}

// --- Count Occurrences ---
int countOccurrences(const vector<int>& arr, int x) {
    return upperBound(arr, x) - lowerBound(arr, x);
}

// --- Search Insert Position ---
int searchInsert(const vector<int>& arr, int x) {
    return lowerBound(arr, x);
}

In [3]:

    vector<int> arr = {1, 2, 2, 2, 3, 4, 4, 5};
   

In [5]:
 cout << "Lower Bound (2):    " << lowerBound(arr, 2)     << " (Expected: 1)" << endl;
    cout << "Upper Bound (2):    " << upperBound(arr, 2)     << " (Expected: 4)" << endl;
    cout << "First Occur (2):    " << firstOccurrence(arr, 2) << " (Expected: 1)\n";
    cout << "Last Occur (2):     " << lastOccurrence(arr, 2)  << " (Expected: 3)\n";
    cout << "Count (2):          " << countOccurrences(arr, 2)<< " (Expected: 3)\n";
    cout << "Insert Position (0): " << searchInsert(arr, 0)    << " (Expected: 0)\n";
    cout << "Insert Position (6): " << searchInsert(arr, 6)    << " (Expected: 8)\n";

Lower Bound (2):    1 (Expected: 1)
Upper Bound (2):    4 (Expected: 4)
First Occur (2):    1 (Expected: 1)
Last Occur (2):     3 (Expected: 3)
Count (2):          3 (Expected: 3)
Insert Position (0): 0 (Expected: 0)
Insert Position (6): 8 (Expected: 8)


No documentation found for o


# Category 1 — Single Element in Sorted Array & Peak Element (1D)

---

## Single Element in Sorted Array

**Setup:** Every element appears twice except one. Array is sorted.

**The Pair Invariant:**
In an undisturbed array, pairs sit at **(even, odd)** index pairs: $(0,1), (2,3), (4,5)\ldots$

The single element **breaks this pairing** at its position. So:

$$\text{if } arr[mid] == arr[mid \oplus 1] \Rightarrow \text{single is to the RIGHT, go } low = mid + 1$$
$$\text{if } arr[mid] \neq arr[mid \oplus 1] \Rightarrow mid \text{ could be the single, go } high = mid$$

**The delta — why $mid \oplus 1$?**

`mid ^ 1` is the elegant parity flip:
- If `mid` is even → `mid ^ 1 = mid + 1` (checks right neighbor, its pair partner)
- If `mid` is odd → `mid ^ 1 = mid - 1` (checks left neighbor, its pair partner)

This single XOR replaces the entire `if mid % 2 == 0` branch. One line, zero branches.

**Search Space Invariant:**
$$\forall j < low: \text{all pairs intact} \qquad \text{single} \in [low, high]$$

**Why `high = mid` (not `mid - 1`)?**
Because `mid` itself could be the answer — you can't eliminate it. This is a `lo < hi` loop (not `lo <= hi`), terminating when `lo == hi` = the answer.

---

## Peak Element (1D)

**Definition:** $arr[peak] > arr[peak-1]$ and $arr[peak] > arr[peak+1]$. Any peak is acceptable.

**The gradient trick:** You don't need to find *the* peak — just *a* peak. This is the unlock.

**Binary decision at `mid`:**

$$arr[mid] < arr[mid+1] \Rightarrow \text{ascending slope} \Rightarrow \text{peak lies RIGHT} \Rightarrow low = mid + 1$$
$$arr[mid] > arr[mid+1] \Rightarrow \text{descending slope} \Rightarrow mid \text{ could be peak} \Rightarrow high = mid$$

**Why this works (the guarantee):**
The array boundaries act as $-\infty$. So:
- If you're on an ascending slope, the peak is to the right — **guaranteed** because the array must come back down somewhere before $-\infty$.
- If descending, peak is at or to the left of `mid`.

**Why `high = mid` (not `mid - 1`)?**
Same reason as Single Element — `mid` is a candidate. Eliminating it risks losing the answer.

**Loop invariant:** Use `lo < hi`. When `lo == hi`, that index is the peak.

---

## The Structural Pattern Behind Both

Both problems share the **same BS skeleton** — and it's different from lb/ub:

| | lb / ub | Single Element / Peak |
|---|---|---|
| Loop condition | `lo <= hi` | `lo < hi` |
| Terminal | `ans` variable | `lo == hi` is the answer |
| Why | Candidate saved separately | `mid` itself could be answer, can't eliminate |
| `high = mid` | Never | Always on candidate side |

In [7]:
// ─── Single Element in Sorted Array ───────────────────────────────────────
// Key: mid ^ 1 flips parity — checks the "pair partner" of mid
int singleNonDuplicate(vector<int>& arr) {
    int lo = 0, hi = (int)arr.size() - 1;
    while (lo < hi) {
        int mid = lo + (hi - lo) / 2;
        // mid ^ 1: if mid even → mid+1, if mid odd → mid-1
        if (arr[mid] == arr[mid ^ 1]) lo = mid + 1; // pair intact → single is RIGHT
        else                          hi = mid;       // pair broken → single is HERE or LEFT
    }
    return arr[lo]; // lo == hi == answer
}

// ─── Peak Element (1D) ────────────────────────────────────────────────────
// Key: follow the ascending slope — a peak is guaranteed in that direction
int findPeakElement(vector<int>& arr) {
    int lo = 0, hi = (int)arr.size() - 1;
    while (lo < hi) {
        int mid = lo + (hi - lo) / 2;
        if (arr[mid] < arr[mid + 1]) lo = mid + 1; // ascending → peak is RIGHT
        else                         hi = mid;       // descending → peak is HERE or LEFT
    }
    return lo; // lo == hi == peak index
}

In [8]:
vector<int> a = {1, 1, 2, 2, 3, 3, 4, 8, 8};
    cout << "Single element: " << singleNonDuplicate(a) << "\n"; // 4

    vector<int> b = {3, 3, 7, 7, 10, 11, 11};
    cout << "Single element: " << singleNonDuplicate(b) << "\n"; // 10

    // Peak Element
    vector<int> c = {1, 2, 3, 1};
    cout << "Peak index: " << findPeakElement(c) << "\n"; // 2

    vector<int> d = {1, 2, 1, 3, 5, 6, 4};
    cout << "Peak index: " << findPeakElement(d) << "\n"; // 1 or 5 (any valid peak)

Single element: 4
Single element: 10
Peak index: 2
Peak index: 5
